# Automated Interior Design System — production pipeline

This notebook **drives the `ids` package**; it does not reimplement it. Every
cell calls the same code the FastAPI service calls, so anything verified here is
verified in production.

```
ids/
  scene.py      shared scene model + hard-constraint validator
  solver.py     Workflow 2 — CP-SAT (optional) or sweep, then annealing
  models.py     Workflow 1 — FP-Growth, style embeddings, price GBR
  history.py    order-history adapters (real dataframe + synthetic bootstrap)
  catalog.py    styles, palettes, tiers, SKU dimensions
  textures.py   procedural PBR texture bakery
  export.py     Workflow 3 — single-file HTML assembly
  pipeline.py   orchestration
  service.py    FastAPI surface
```

Mapping to the architecture doc:

| Doc | Module | Notebook |
|---|---|---|
| **W1** Associative / recommendation | `ids.models`, `ids.history` | §2 |
| **Shared** Spatial constraint solver | `ids.solver`, `ids.scene` | §3 |
| **W3a** 2D rendering | SVG travels with the scene | §5 |
| **W3b** 3D rendering | `ids.export` | §4–§5 |

Install: `pip install -e ".[dev]"`, optionally `".[solver]"` for OR-Tools and
`".[serve]"` for the API.

## 1 · Setup and catalogue

In [ ]:
import json, sys, time
from pathlib import Path

sys.path.insert(0, ".")            # or: pip install -e .
from ids import (Brief, Catalog, DesignPipeline, Scene, SpatialSolver,
                 SweepBackend, validate)
from ids.history import from_dataframe, synthesise

BLOBS = Path("build/data_blobs.js")
ARTIFACTS = Path("artifacts") / time.strftime("%Y-%m-%d")

catalog = Catalog.from_blobs(BLOBS)
print("styles   :", list(catalog.styles))
print("woods    :", list(catalog.woods))
print("fabrics  :", list(catalog.fabrics))
print("budgets  :", catalog.budgets)
print("variants :", len(catalog.variants), "| SKUs:", len(catalog.skus))
print("default  :", catalog.default_brief())

## 2 · Workflow 1 — the associative layer

**Swap in real data here.** `synthesise()` exists so the pipeline runs from a
cold start; `from_dataframe()` is the production path and expects one row per
order line with the order-level attributes repeated:

```python
import pandas as pd
orders_df = pd.read_sql("select * from order_lines", conn)
history = from_dataframe(orders_df)     # nothing downstream changes
```

Anything trained on synthetic data carries that fact into the bundle metadata,
so you can tell from the artefact which one produced it.

In [ ]:
history = synthesise(9000, seed=7)
print(f"{len(history)} bundles, synthetic={history.synthetic}")
print("median basket:", sorted(len(b) for b in history.baskets)[len(history) // 2])
print("example      :", history.baskets[0])

### Train

Three models, one bundle. FP-Growth mines conditional pattern bases *weighted* —
the textbook version that duplicates each path `count` times is exponential on
real basket sizes and will hang.

The style embedder is content-based (TF-IDF over tag documents → SVD). The doc
specifies a **vision** model over product imagery; that is a drop-in via
`StyleEmbedder.fit_images(encoder, images)` — everything downstream consumes
vectors, not the encoder.

In [ ]:
t = time.time()
pipe = DesignPipeline.build(blobs=BLOBS, history=history, version="v1")
print(f"trained in {time.time() - t:.1f}s")
print(json.dumps(pipe.bundle.export()["meta"], indent=2))

print("\ntop association rules by lift:")
for r in pipe.bundle.association.rules[:10]:
    print(f"  {{{', '.join(r.antecedent)}}} -> {r.consequent:14s}"
          f" conf {r.confidence:.2f}  lift {r.lift:.2f}")

In [ ]:
pipe.save(ARTIFACTS)
print("artefacts:", sorted(p.name for p in ARTIFACTS.iterdir()))

# reload without retraining — this is what the service does at boot
reloaded = DesignPipeline.load(ARTIFACTS, blobs=BLOBS)
print("reloaded rules:", len(reloaded.bundle.association.rules))

### The exported price model must match sklearn

The browser evaluates the trees itself, so a mismatch means the customer sees a
different number from the one your server quotes. This is a CI gate
(`test_price_parity`), not a nicety.

In [ ]:
from ids.models import PriceModel

exported = pipe.bundle.price.export()
worst = 0.0
for o in history.orders[:200]:
    x = pipe.bundle.price.featurise(o["items"], o["bhk"], o["style"], o["tier"], o["city"])
    a = PriceModel.eval_exported(exported, x)
    b = pipe.bundle.price.predict(o["items"], o["bhk"], o["style"], o["tier"], o["city"])
    worst = max(worst, abs(a - b) / b)
print(f"worst relative disagreement over 200 orders: {worst:.2e}")
print(f"exported size: {len(json.dumps(exported)) / 1024:.0f} KB, "
      f"{len(exported['trees'])} trees")

## 3 · Shared step — the spatial constraint solver

Both rendering workflows need the same validated positions, so this is the only
place coordinates are decided.

Two stages. **Feasibility** places every item at the lowest-cost pose satisfying
all hard constraints — `CPSATBackend` when `ortools` is importable, `SweepBackend`
otherwise; both emit the same `Placement` list. **Polish** is simulated annealing
on the soft objective (open space, functional adjacency, wall alignment, visual
balance), carrying hard constraints as a large penalty so the chain can cross
infeasible ground, and returning the best *feasible* state seen.

Placement rules live in `ids.solver.RULES` — one `Rule` per category saying how
it wants to sit. That table is where you tune layout behaviour.

In [ ]:
from ids.solver import RULES, rule_for

for cat in ["bed", "sofa", "counter_run", "rug", "floor_lamp"]:
    r = rule_for(cat)
    print(f"{cat:14s} anchor={r.anchor:8s} clearance={r.clearance:.2f}m "
          f"beside={r.against or '-'} priority={r.priority}")

In [ ]:
solver = SpatialSolver(backend=SweepBackend(), seed=7, iterations=900)
scene = Scene.from_dict(catalog.variant_scene("3 BHK", "Standard"))

t = time.time()
solved, reports = solver.solve_scene(scene)
print(f"solved {len(solved.objects)} objects in {time.time() - t:.2f}s\n")
for r in reports:
    print(" ", r)
print("\nhard-constraint violations:", len(validate(solved)))

### Gate every variant before shipping

The browser must never load a layout the solver has not validated.

In [ ]:
t = time.time()
pipe.solve_all()
print(f"re-solved {len(pipe.solve_summary)} variants in {time.time() - t:.1f}s")
bad = {k: v for k, v in pipe.solve_summary.items() if v}
print("variants with violations:", bad or "none")
assert not bad, "a variant failed validation - do not ship this build"


### Inspect a solved room

`validate()` is the same function the solver uses as its acceptance test and CI
uses as a gate, so this plot and the gate can never disagree.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

def draw(scene, room_id, ax=None):
    room = scene.room(room_id)
    ax = ax or plt.subplots(figsize=(6, 6))[1]
    x, y, w, d = room.rect
    ax.add_patch(Rectangle((x, y), w, d, fill=False, lw=2, ec="#333"))
    for op in scene.openings_in(room_id):
        ax.plot([op.p0[0], op.p1[0]], [op.p0[1], op.p1[1]],
                lw=4, color="#c2410c" if op.is_door else "#2563eb")
        sb = op.swing_box()
        if sb:
            ax.add_patch(Rectangle((sb.x0, sb.y0), sb.w, sb.d, fc="#c2410c", alpha=.10))
    for o in scene.objects_in(room_id):
        b = o.footprint()
        ax.add_patch(Rectangle((b.x0, b.y0), b.w, b.d, fc="#8ea9c9", ec="#31506e", alpha=.85))
        ax.text(b.cx, b.cy, o.category.replace("_", "\n"), ha="center", va="center", fontsize=6)
    ax.set_aspect("equal"); ax.set_title(f"{room.label} — {room_id}"); ax.autoscale_view()
    return ax

fig, axes = plt.subplots(1, 2, figsize=(13, 6))
draw(scene, "living_room", axes[0]); axes[0].set_title("cached layout")
draw(solved, "living_room", axes[1]); axes[1].set_title("solver output")
plt.tight_layout(); plt.show()

## 4 · Textures and the 3D viewer

Albedo maps are baked as **low-contrast detail multipliers** (mean ≈ 0.87), not
as finished colours. The viewer sets `material.color` from the palette and lets
the map modulate it, so the colour / wood / fabric toggles survive the multiply.
An earlier build divided the colour by the map mean to cancel the darkening —
that clamped at 255 and washed every warm surface out to white. The residual
~13% darkening is paid back once, globally, in `toneMappingExposure`.

In [ ]:
from ids import textures as bakery

if not Path("build/textures.json").exists():
    bakery.main()                       # ~40s
atlas = json.loads(Path("build/textures.json").read_text())
print(f"{len(atlas['maps'])} maps, "
      f"{sum(len(v) for v in atlas['maps'].values()) / 1024:.0f} KB of base64")
print("albedo means:", atlas["means"])

In [ ]:
import base64, io
from PIL import Image, ImageDraw

names = [n for n in sorted(atlas["maps"]) if n.endswith("_map")]
cols, th = 5, 170
sheet = Image.new("RGB", (cols * th, ((len(names) + cols - 1) // cols) * (th + 16)),
                  (235, 232, 226))
d = ImageDraw.Draw(sheet)
for i, n in enumerate(names):
    im = Image.open(io.BytesIO(base64.b64decode(atlas["maps"][n].split(",")[1])))
    x, y = (i % cols) * th, (i // cols) * (th + 16)
    sheet.paste(im.convert("RGB").resize((th, th)), (x, y))
    d.text((x + 3, y + th + 2), n, fill=(30, 30, 30))
sheet

## 5 · Design a home and ship the viewer

`design()` is the whole system in one call: validate the brief → resolve tier and
palette → solve the layout → rank add-ons → predict the price.

In [ ]:
brief = Brief(bhk="3 BHK", style="Luxury", budget="₹12L–₹20L", quality="Premium",
              wood="Walnut Laminate", fabric="Velvet",
              colors=("Off White", "Charcoal Grey", "Champagne Gold"))

result = pipe.design(brief)
print(json.dumps(result.summary(), indent=2, ensure_ascii=False))

print("\nranked add-ons:")
for r in result.recommendations:
    rule = r["rule"]
    print(f"  {r['category']:16s} score {r['score']:.3f}  "
          f"conf {rule['confidence']:.2f}  lift {rule['lift']:.2f}  "
          f"style fit {r['style_fit']:.0%}  from {{{', '.join(rule['antecedent'])}}}")

In [ ]:
out = pipe.render_html(result, "out/home_viewer.html",
                       textures="build/textures.json",
                       viewer_js="build/viewer.js")
print(f"{out}  ({out.stat().st_size / 1024 / 1024:.2f} MB)")

## 6 · Serve it

```bash
IDS_ARTIFACTS=artifacts/2026-09-11 IDS_BLOBS=build/data_blobs.js \
  uvicorn ids.service:app --host 0.0.0.0 --port 8000
```

`design` is the only heavy endpoint — annealing dominates it. It is CPU-bound and
holds the GIL, so scale with worker processes, not threads.

In [ ]:
from ids.service import build_app

try:
    app = build_app()
    print("routes:", sorted(r.path for r in app.routes if hasattr(r, "path")))
except Exception as e:
    print("fastapi not installed — pip install -e '.[serve]'  (", type(e).__name__, ")")

## 7 · Gates

Run `pytest -q` in CI. The two that matter:

* `test_solver_feasible_all_variants` — no layout ever ships with a hard
  constraint breach.
* `test_price_parity` — the exported trees reproduce sklearn, so browser and
  server quote the same number.

In [ ]:
import subprocess
print(subprocess.run([sys.executable, "-m", "pytest", "-q"],
                     capture_output=True, text=True).stdout[-2000:])

## Known limitations

These are real, and worth carrying into planning rather than discovering later.

* **The price GBR is unconstrained**, so it is not monotonic in item count — a
  superset basket can price marginally lower. `design()` clamps this;
  `test_price_monotonicity_rate` keeps the raw rate visible. If that rate climbs,
  the feature set needs work, not a bigger clamp.
* **The style embedder is content-based**, not the vision model the doc
  specifies. Swap via `StyleEmbedder.fit_images`.
* **Floor-plan geometry is input, not generated.** The solver places furniture
  within a given plan; rooms, walls and openings come from the property.
* **The 3D furniture is procedural**, a stand-in for a glTF library. The scene
  schema already carries `asset_url` per object and the viewer prefers a real GLB
  whenever one loads. The doc is right that this asset pipeline "tends to be
  underestimated" — it is the largest remaining piece of work.
* **Nothing here is a generative image model.** No single HTML file can render
  photoreal imagery from a diffusion model client-side. Realism comes from
  procedural PBR textures, image-based lighting and geometry.